In [2]:
# Cell 1 (setup; use whatever PROJECT_ROOT cell your other notebooks have)
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
from src import diagnostics as dg

In [3]:
# Cell 2: files, features, split sizes (seconds)
feats = dg.list_inputs()

=== Source files ===
  Monday   : Monday-WorkingHours.pcap_ISCX.csv
  Wednesday: Wednesday-workingHours.pcap_ISCX.csv

=== Retained features: 76 ===
   1. Flow Duration
   2. Total Fwd Packets
   3. Total Backward Packets
   4. Total Length of Fwd Packets
   5. Total Length of Bwd Packets
   6. Fwd Packet Length Max
   7. Fwd Packet Length Min
   8. Fwd Packet Length Mean
   9. Fwd Packet Length Std
  10. Bwd Packet Length Max
  11. Bwd Packet Length Min
  12. Bwd Packet Length Mean
  13. Bwd Packet Length Std
  14. Flow Bytes/s
  15. Flow Packets/s
  16. Flow IAT Mean
  17. Flow IAT Std
  18. Flow IAT Max
  19. Flow IAT Min
  20. Fwd IAT Total
  21. Fwd IAT Mean
  22. Fwd IAT Std
  23. Fwd IAT Max
  24. Fwd IAT Min
  25. Bwd IAT Total
  26. Bwd IAT Mean
  27. Bwd IAT Std
  28. Bwd IAT Max
  29. Bwd IAT Min
  30. Fwd PSH Flags
  31. Bwd PSH Flags
  32. Fwd URG Flags
  33. Bwd URG Flags
  34. Fwd Header Length
  35. Bwd Header Length
  36. Fwd Packets/s
  37. Bwd Packets/s
  38. Min Pac

In [4]:
# Cell 3: Destination Port only (seconds)
_ = dg.port_only_check()

=== Destination Port ONLY (RF, 50 trees) - AUROC 0.9494 ===
  distinct score values on val: 3
  target FPR  achieved FPR   recall  benign tied @thr
          1%        0.0000   0.0000            0.1011
          5%        0.0000   0.0000            0.1011
         10%        0.0000   0.0000            0.1011
         20%        0.1011   1.0000            0.8989

  Top destination ports by share of each class (val):
    benign: 53=42.2%, 443=24.4%, 80=10.1%, 123=1.0%, 22=0.3%
    attack: 80=100.0%, 444=0.0%
  If benign port 80 > 1% and attack port 80 is high, the 1% threshold sits
  on port 80's score, strict `>` flags nothing, and 0% recall is a tie artifact.


In [5]:
# Cell 4: RF leakage and feature-drop check (a few minutes)
_ = dg.rf_leakage_check()

--- baseline RF (all features) ---

=== Top 10 feature importances ===
Bwd Packet Length Std    0.0966
Packet Length Std        0.0870
Packet Length Variance   0.0848
Bwd Packet Length Mean   0.0740
Bwd Packet Length Max    0.0594
Average Packet Size      0.0565
Avg Bwd Segment Size     0.0499
Fwd IAT Std              0.0452
Max Packet Length        0.0425
Packet Length Mean       0.0364

--- retrain without top 1: ['Bwd Packet Length Std'] ---

--- retrain without top 3: ['Bwd Packet Length Std', 'Packet Length Std', 'Packet Length Variance'] ---

--- retrain without top 5: ['Bwd Packet Length Std', 'Packet Length Std', 'Packet Length Variance', 'Bwd Packet Length Mean', 'Bwd Packet Length Max'] ---

=== RF with top features removed (val) ===
     variant  auroc    thr  achieved_fpr  recall@1%
all features 1.0000 0.0050        0.0092     1.0000
  drop top 1 1.0000 0.0050        0.0094     1.0000
  drop top 3 1.0000 0.0050        0.0096     1.0000
  drop top 5 1.0000 0.0050        0.00

In [6]:
# Cell 5: with vs without Destination Port, both models (~5 min, mostly AE fits)
abl = dg.context_ablation(seeds=(42,))


--- RF-IDS | without port (current) | seed 42 ---

--- Autoencoder | without port (current) | seed 42 ---
  epoch   1/30  train MSE: 0.232524
  epoch   5/30  train MSE: 0.051120
  epoch  10/30  train MSE: 0.037277
  epoch  15/30  train MSE: 0.036031
  epoch  20/30  train MSE: 0.025630
  epoch  25/30  train MSE: 0.021295
  epoch  30/30  train MSE: 0.015909

--- RF-IDS | with port | seed 42 ---

--- Autoencoder | with port | seed 42 ---
  epoch   1/30  train MSE: 0.250941
  epoch   5/30  train MSE: 0.054652
  epoch  10/30  train MSE: 0.043120
  epoch  15/30  train MSE: 0.035911
  epoch  20/30  train MSE: 0.031843
  epoch  25/30  train MSE: 0.020754
  epoch  30/30  train MSE: 0.017423

=== Context ablation (val) ===
      model  seed                variant  auroc    thr  achieved_fpr  recall@1%
     RF-IDS    42 without port (current) 1.0000 0.0050        0.0092     1.0000
Autoencoder    42 without port (current) 0.9557 0.1731        0.0100     0.8423
     RF-IDS    42              with 

In [7]:
# Cell 6: missed-flow and false-positive analysis (~2 min)
_ = dg.error_analysis()

--- Autoencoder ---
  epoch   1/30  train MSE: 0.232524
  epoch   5/30  train MSE: 0.051120
  epoch  10/30  train MSE: 0.037277
  epoch  15/30  train MSE: 0.036031
  epoch  20/30  train MSE: 0.025630
  epoch  25/30  train MSE: 0.021295
  epoch  30/30  train MSE: 0.015909

AE threshold 0.1731, achieved FPR 0.0100
attack                  n  recall  median score / thr
DoS GoldenEye        1542   0.456                0.89
DoS Hulk            25927   0.876                2.52
DoS Slowhttptest      784   0.782                2.01
DoS slowloris         807   0.560                2.56
Heartbleed              1   1.000               12.18

  DoS GoldenEye: 839 missed. Largest median gaps vs caught same-type flows:
    Packet Length Variance              -9.52 sd
    Bwd Packet Length Mean              -5.10 sd
    Avg Bwd Segment Size                -5.10 sd
    Bwd Packet Length Std               -5.00 sd
    Packet Length Std                   -3.71 sd
    Bwd Packet Length Max               

In [16]:
from pathlib import Path
roots = [Path.home() / d for d in ("Downloads", "Documents", "Desktop")]
for r in roots:
    for h in sorted(r.glob("**/*TrafficLabelling*")) + sorted(r.glob("**/GeneratedLabelledFlows*")):
        print(h)

/Users/skishore/Downloads/TrafficLabelling 
/Users/skishore/Downloads/GeneratedLabelledFlows.md5
/Users/skishore/Downloads/GeneratedLabelledFlows.zip


In [18]:
from pathlib import Path
for p in sorted(Path("/Users/skishore/Downloads").glob("TrafficLabelling*/*.csv")):
    print(p)

/Users/skishore/Downloads/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Tuesday-WorkingHours.pcap_ISCX.csv
/Users/skishore/Downloads/TrafficLabelling /Wednesday-workingHours.pcap_ISCX.csv


In [23]:
import importlib, src.row_order_check as rc
importlib.reload(rc)
rc.check_row_order("data/raw/Monday-WorkingHours.pcap_ISCX.csv",
                   "/Users/skishore/Downloads/TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv")

MachineLearningCSV rows: 529,918   labelled-flows rows: 529,918

Alignment (share of rows where the shared column matches exactly):
  Destination Port          100.0000%
  Flow Duration             100.0000%
  Total Fwd Packets         100.0000%
  Total Backward Packets    100.0000%
  -> files align row-for-row

Timestamps parsed: 100.00%
Time range after 12-hour fix: 08:55 -> 17:01

WHOLE FILE, row order vs time:
  Spearman(row index, time)       : 0.4531
  consecutive pairs non-decreasing: 60.59%
  backward jumps larger than 5 min: 144,250

PER LABEL (our split cuts each label group by row position):
  label                  rows  Spearman  non-decr.  drops>5min  time span
  BENIGN               529918     0.453     60.59%     144,250  08:55-17:01

Time span of each tenth of the file, in row order:
  rows       0- 52,991: 08:55 -> 10:13
  rows  52,991-105,983: 08:57 -> 11:12
  rows 105,983-158,975: 08:56 -> 12:07
  rows 158,975-211,967: 08:56 -> 13:38
  rows 211,967-264,959: 08:56 ->

{'spearman': 0.45314769106173464,
 'nondecreasing': 0.6058873370735417,
 'drops_gt_5min': 144250}